<a href="https://colab.research.google.com/github/chrispchen/ChenHypoxia/blob/main/DifferentialPolyadenylation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Differential Polyadenylation

## Mount and load

In [1]:
# Mount google drive

from google.colab import drive
drive.mount('/content/drive/')
data_path = '/your/patj/'  #change dir to your project folder

Mounted at /content/drive/


In [2]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 27.0 MB/s eta 0:00:00


In [3]:
# import libraries
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from glob import glob
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [4]:
# import the polyA data table
df = pd.read_csv('/your/path/CombinedR1R2_table_annotated.csv', sep = '\t')
df

/tmp/ipykernel_461/3163925229.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Chris/Nano3pSeq/PostPolyTailor/CombinedR1R2_table_annotated.csv', sep = '\t')


,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,exons,additional,comment,Gene,Group,Genotype,Method,Condition,Batch,Adjusted_pt_length
0,b08e7e33-7610-48f3-8ed9-f1a1293ed16f,SQK-NBD114-24_barcode07,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,39132-39280,gene_assignment=unique; PolyA=False; Classific...,NaN,XLOC_043623,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1,2.000
1,1b514c91-eccd-4fe6-b857-1093e4061fcd,SQK-NBD114-24_barcode11,60,not_continuous,-0.2,3.9,100,101,GAGGCAGAGG,TTTTT,...,2235-3171,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_2a,Tent5ba42,Hypoxia18,Hypoxia18,Batch1,1.750
2,8ab9bff2-43b2-4696-a238-b6c72f444674,SQK-NBD114-24_barcode07,18,no_primer,0.0,0.0,0,0,NaN,NaN,...,2866-3405,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1,2.000
3,7f1adb2e-6d1c-483d-8bfc-037643264136,SQK-NBD114-24_barcode08,49,not_continuous,-0.2,2.6,144,149,CAGAGTCTGA,TTTT,...,2877-3141,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_2a,Wildtype,Hypoxia18,Hypoxia18,Batch1,1.750
4,4dd554ec-22ab-4fc3-84d9-0ed13e4c9fa3,SQK-NBD114-24_barcode07,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,3095-3559,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1,2.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27307111,eddc7366-6312-47a7-b918-5beedcc5abe7,SQK-NBD114-24_barcode13,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,16399-16596,gene_assignment=inconsistent_non_intronic; Pol...,NaN,NaN,Hyp_rRNA_WT_1,Wildtype,Hypoxia18,Hypoxia18,Batch2,2.000
27307112,cdbb438e-243c-4122-a166-959b0145e540,SQK-NBD114-24_barcode05,60,OK,10.8,3.5,104,104,CAGAGCAGAG,TTTTTTTTTTTTTTTTTATT,...,16400-16515,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_2,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2,15.500
27307113,a25e139b-ff92-4766-8e3b-86a37959dea8,SQK-NBD114-24_barcode16,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,16402-16560,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2,2.000
27307114,4d397346-f635-44df-b346-b6588a666674,SQK-NBD114-24_barcode04,60,OK,12.9,2.9,103,103,CAGAGCAGAG,TTATTTTTTTTTTTTTT,...,16403-16522,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_1,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2,18.125


## Filter by samples for comparison

In [5]:
# filter by conditinos for comparison


df = df[df["filter"] == 'OK']

df = df[df["Gene"].notna()]

df = df.loc[df['Group'].isin((
    "Norm_rRNA_WT_1","Norm_rRNA_WT_2", "Norm_rRNA_WT_3",
    "Hyp_rRNA_WT_1a", "Hyp_rRNA_WT_2a", "Hyp_rRNA_WT_3a",
    "Hyp_rRNA_WT_1", "Hyp_rRNA_WT_2", "Hyp_rRNA_WT_3"
))]


## BH adjusted Mann-Whitney

In [7]:

# define function for stats

def run_genotype_analysis(df, min_reads_per_group=20, output_filename=None):
    """
    Perform Mann-Whitney U test

    Parameters:
    -----------
    df : DataFrame
        Input dataframe with columns: Gene, Genotype, Adjusted_pt_length
    min_reads_per_group : int
        Minimum number of reads required per genotype group (default: 20)
    output_filename : str, optional
        Path to save results CSV

    Returns:
    --------
    results_dict : dict
        Dictionary containing results tables and metadata
    """
    # Drop rows with missing Adjusted_pt_length once
    df_clean = df.dropna(subset=['Adjusted_pt_length'])

    # Pre-compute genotype masks
    Normoxia_mask = df_clean['Condition'] == 'Normoxia6hpf'
    Hypoxia_mask = df_clean['Condition'] == 'Hypoxia18'

    # Group by gene
    grouped = df_clean.groupby('Gene', sort=False)

    # Pre-allocate list for results
    results = []

    for gene, gene_data in grouped:
        # Use boolean indexing on pre-grouped data
        Normoxia_values = gene_data.loc[Normoxia_mask[gene_data.index], 'Adjusted_pt_length'].values
        Hypoxia_values = gene_data.loc[Hypoxia_mask[gene_data.index], 'Adjusted_pt_length'].values

        # Skip if either group has fewer than minimum value
        if len(Normoxia_values) < min_reads_per_group or len(Hypoxia_values) < min_reads_per_group:
            continue

        # Calculate means
        mean_Normoxia = Normoxia_values.mean()
        mean_Hypoxia = Hypoxia_values.mean()

        # Perform Mann-Whitney U test
        stat, p_value = mannwhitneyu(Normoxia_values, Hypoxia_values, alternative='two-sided')

        # Store results
        results.append({
            'Gene': gene,
            'Avg_N6WT_Adjusted_pt_length': mean_Normoxia,
            'Avg_H18WT_Adjusted_pt_length': mean_Hypoxia,
            'p_value': p_value,
            'n_N6WT': len(Normoxia_values),
            'n_H18WT': len(Hypoxia_values)
        })

    # Convert to DataFrame
    result_df = pd.DataFrame(results)

    if len(result_df) == 0:
        print(f"No genes met the minimum threshold of {min_reads_per_group} reads per group")
        return {
            'results_table': pd.DataFrame(),
            'results_no_na': pd.DataFrame(),
            'min_reads': min_reads_per_group,
            'n_genes_tested': 0
        }

    # Adjust p-values using Benjamini-Hochberg (FDR)
    # Each threshold gets its own FDR correction based on its gene set
    rejected, p_adj, _, _ = multipletests(result_df['p_value'], method='fdr_bh')
    result_df['BH_adjusted_p_value'] = p_adj

    # Vectorized calculations
    result_df["log2FC_H18WT_vs_N6WT"] = np.log2(
        result_df["Avg_H18WT_Adjusted_pt_length"] /
        result_df["Avg_N6WT_Adjusted_pt_length"]
    )
    result_df["delta_pT_H18WT_vs_N6WT"] = (
        result_df["Avg_H18WT_Adjusted_pt_length"] -
        result_df["Avg_N6WT_Adjusted_pt_length"]
    )

    # Sort by adjusted p-value
    result_df = result_df.sort_values('BH_adjusted_p_value')

    # Remove rows with NaN values
    result_df_no_na = result_df.dropna()

    # Save
    if output_filename:
        result_df_no_na.to_csv(output_filename, index=False)
        print(f"Results saved to: {output_filename}")

    return {
        'results_table': result_df,
        'results_no_na': result_df_no_na,
        'min_reads': min_reads_per_group,
        'n_genes_tested': len(result_df),
        'n_genes_no_na': len(result_df_no_na)
    }


#Clean the dataframe once before all analyses
df_clean = df.dropna(subset=['Adjusted_pt_length'])

# Run analyses with the cleaned dataframe (no need for .copy())
print("Running analyses.......")

results_20 = run_genotype_analysis(
    df_clean,
    min_reads_per_group=20,
    output_filename='/your/path/ByCondition_20reads.csv'
)


print(f"Genes tested (20 reads): {results_20['n_genes_tested']}")

# Filter for significant genes
print("\nSignificant genes (BH p < 0.05, |delta| > 5):")

sig_genes_20 = results_20['results_no_na'][
    (results_20['results_no_na']['BH_adjusted_p_value'] < 0.05) &
    (results_20['results_no_na']['delta_pT_H18WT_vs_N6WT'].abs() > 5)
]
print(f"  20 reads: {len(sig_genes_20)}")


# View example results
print("\nTop results (20 reads):")
results_20['results_no_na'].head()

Running analyses.......
Results saved to: /content/drive/MyDrive/Chris/Nano3pSeq/N6WTvsH18WT_Stats/ByCondition_20reads.csv
Genes tested (20 reads): 6954

Significant genes (BH p < 0.05, |delta| > 5):
  20 reads: 3636

Top results (20 reads):


,Gene,Avg_N6WT_Adjusted_pt_length,Avg_H18WT_Adjusted_pt_length,p_value,n_N6WT,n_H18WT,BH_adjusted_p_value,log2FC_H18WT_vs_N6WT,delta_pT_H18WT_vs_N6WT
181,hmgb2a,131.362889,76.989159,0.0,7689,11899,0.0,-0.770831,-54.373730
2784,anp32e,126.499918,65.542837,0.0,3035,5979,0.0,-0.948626,-60.957080
761,nucks1a,138.143071,83.067442,0.0,2172,2782,0.0,-0.733808,-55.075628
768,si:ch211-222l21.1,126.770815,83.689853,0.0,3357,6455,0.0,-0.599098,-43.080962
1506,BX088707.1,97.156291,56.092939,0.0,3795,10176,0.0,-0.792488,-41.063352
